# Delhi Colonies Public Services Index (25 August 2020)

## Compute the following indices:
* Index with touching neighbors [effective service count divided by population]
* Index with bounding box neighbors [effective service count divided by population]
* Index with touching neighbors [effective service count divided by population/area]
* Index with bounding box neighbors [effective service count divided by population/area]


### How to compute indices
* Load in colonies datasets from Pickle **[done]**
* Import services shapefiles **[done]**
    * Make sure correct file paths exist
    * Ensure that all shapefiles are valid using `check_shapefile` function
    * Reproject shapefiles to EPSG 7760 (if needed)
* Compute all Services indices (turn into a function)
    * Touching neighbors, Population Size
    * Touching neighbors, Population Density
    * bbox neighbors, Population Size
    * bbox neighbors, Population Density

## Import modules and set constants

In [ ]:
import os
import pickle
from importlib import reload
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, box
import spatial_index_utils

In [ ]:
reload(spatial_index_utils)

In [ ]:
# WGS 84 / Delhi
epsg_code = 7760

## Import Colonies Datasets

In [ ]:
with open('colonies_touch_nbrs25Aug2020.pkl', 'rb') as f:
    colonies_touch_nbrs = pickle.load(f)
    
colonies_touch_nbrs.head()

In [ ]:
with open('colonies_bbox_nbrs25Aug2020.pkl', 'rb') as f:
    colonies_bbox_nbrs = pickle.load(f)
    
colonies_bbox_nbrs.head()

## Import services shapefiles

In [ ]:
# Define filepaths

services_dir = os.path.join('shapefiles', 'Spatial_Index_GIS', 'Public Services')

bank_fp = os.path.join(services_dir, 'Banking', 'Banking.shp')
health_fp = os.path.join(services_dir, 'Health', 'Health.shp')
road_fp = os.path.join(services_dir, 'Major Road', 'Road.shp')
police_fp = os.path.join(services_dir, 'Police', 'Police Station.shp')
ration_fp = os.path.join(services_dir, 'Ration', 'Ration.shp')
school_fp = os.path.join(services_dir, 'School', 'schools7760.shp')
transport_fp = os.path.join(services_dir, 'Transport', 'Transport.shp')

# boundary of Delhi
delhi_bounds_filepath = os.path.join('shapefiles', 'delhi_bounds_buffer.shp')

# Check that all filepaths exist
filepath_list = [bank_fp, health_fp, road_fp, police_fp, ration_fp, school_fp, transport_fp, delhi_bounds_filepath]

for filepath in filepath_list:
    if not os.path.exists(filepath):
        print('{} does not exist'.format(filepath))

In [ ]:
# Import services
bank = gpd.read_file(bank_fp)
health = gpd.read_file(health_fp)
road = gpd.read_file(road_fp)
police = gpd.read_file(police_fp)
ration = gpd.read_file(ration_fp)
school = gpd.read_file(school_fp)
transport = gpd.read_file(transport_fp)

## Check validity of services shapefiles
* Duplicate rows are okay for ATMs (I assume that ATM locations for the same bank in a similar location will seem to be counted twice)
* Look specifically for invalid geometries and whether shapefile is fully contained within Delhi

In [ ]:
spatial_index_utils.check_shapefile(gdf=bank, gdf_name='bank', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=health, gdf_name='health', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=road, gdf_name='road', 
                                    geom_type='Line', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=police, gdf_name='police', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=ration, gdf_name='ration', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=school, gdf_name='school', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

In [ ]:
spatial_index_utils.check_shapefile(gdf=transport, gdf_name='transport', 
                                    geom_type='Point', 
                                    delhi_bounds_filepath=delhi_bounds_filepath)

## Check CRS (all shapefiles should be in EPSG: 7760)

In [ ]:
bank.crs == health.crs == road.crs == police.crs == ration.crs == school.crs == transport.crs

In [ ]:
bank.crs

In [ ]:
colonies_bbox_nbrs.crs == colonies_touch_nbrs.crs == bank.crs

## Define Point and Line Services

In [ ]:
# Define all point services as dictionary
# makes it easier to calculate all point
# services with one function
point_services = {'bank': bank,
                  'health': health,
                  'police': police,
                  'ration': ration,
                  'school': school,
                  'transport': transport}

line_services = {'road': road}

## Calculate all service indices in one function

In [ ]:
from spatial_index_utils import calc_all_services

In [ ]:
calc_all_services?

### Calculate PSI for touching neighbors using Population Size

In [ ]:
colonies_touch_psi_popsize = calc_all_services(polygon_gdf = colonies_touch_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       calc_pop_density = False,
                                       nbr_dist_colname = 'nbrs_dist_touch')

colonies_touch_psi_popsize = colonies_touch_psi_popsize.rename(columns={'road_count':'road_length'})
colonies_touch_psi_popsize

### Calculate PSI for touching neighbors using Population Density

In [ ]:
colonies_touch_psi_popdensity = calc_all_services(polygon_gdf = colonies_touch_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       calc_pop_density = True,
                                       nbr_dist_colname = 'nbrs_dist_touch')

colonies_touch_psi_popdensity = colonies_touch_psi_popdensity.rename(columns={'road_count':'road_length'})
colonies_touch_psi_popdensity.head()

### Calculate PSI for bbox neighbors using Population Size

In [ ]:
colonies_bbox_psi_popsize = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       calc_pop_density = False,
                                       nbr_dist_colname = 'nbrs_dist_bbox')

colonies_bbox_psi_popsize = colonies_bbox_psi_popsize.rename(columns={'road_count':'road_length'})
colonies_bbox_psi_popsize.head()

### Calculate PSI for touching neighbors using Population Density

In [ ]:
colonies_bbox_psi_popdensity = calc_all_services(polygon_gdf = colonies_bbox_nbrs, 
                                       point_services = point_services, 
                                       line_services = line_services, 
                                       epsg_code = epsg_code, 
                                       calc_pop_density = True,
                                       nbr_dist_colname = 'nbrs_dist_bbox')

colonies_bbox_psi_popdensity = colonies_bbox_psi_popdensity.rename(columns={'road_count':'road_length'})
colonies_bbox_psi_popdensity.head()

## Save Files

In [ ]:
bbox_drop_columns = ['nbrs_bbox', 'nbrs_dist_bbox', 'centroid']
touch_drop_columns = ['nbrs_touch', 'nbrs_dist_touch', 'centroid']

In [ ]:
colonies_bbox_psi_popsize.drop(columns=bbox_drop_columns).to_file('delhi_psi_bbox_popsize_25Aug2020.shp')
colonies_bbox_psi_popdensity.drop(columns=bbox_drop_columns).to_file('delhi_psi_bbox_popdensity_25Aug2020.shp')

In [ ]:
colonies_touch_psi_popsize.drop(columns=touch_drop_columns).to_file('delhi_psi_touch_popsize_25Aug2020.shp')
colonies_touch_psi_popdensity.drop(columns=touch_drop_columns).to_file('delhi_psi_touch_popdensity_25Aug2020.shp')

In [ ]:
colonies_bbox_psi_popsize.to_csv('delhi_psi_bbox_popsize_25Aug2020.csv')
colonies_bbox_psi_popdensity.to_csv('delhi_psi_bbox_popdensity_25Aug2020.csv')

In [ ]:
colonies_touch_psi_popsize.to_csv('delhi_psi_touch_popsize_25Aug2020.csv')
colonies_touch_psi_popdensity.to_csv('delhi_psi_touch_popdensity_25Aug2020.csv')

In [ ]:
with open('colonies_bbox_psi_popsize.pkl', 'wb') as f:
    pickle.dump(colonies_bbox_psi_popsize, f)
    
with open('colonies_bbox_psi_popdensity.pkl', 'wb') as f:
    pickle.dump(colonies_bbox_psi_popdensity, f)

with open('colonies_touch_psi_popsize.pkl', 'wb') as f:
    pickle.dump(colonies_touch_psi_popsize, f)

with open('colonies_touch_psi_popdensity.pkl', 'wb') as f:
    pickle.dump(colonies_touch_psi_popdensity, f)